# Representation analysis: original-space metrics, t-SNE, and shared PCA

This notebook evaluates the four patience-4 ConvNeXtV2 checkpoints used in the paper's post-challenge diagnostic analysis:

- CE-only
- ArcFace-only
- CE+ArcFace (60:40)
- CE+ArcFace (20:80)

It performs three complementary analyses without retraining:

1. quantitative metrics in the original L2-normalized 512-dimensional embedding space;
2. independently fitted t-SNE projections for one held-out fold;
3. orthogonal Procrustes alignment to the CE+ArcFace (60:40) reference space, followed by one shared PCA projection.

The original 512-dimensional metrics are the primary quantitative evidence. The t-SNE panels are supplementary because each projection is fitted independently. The Procrustes/shared-PCA figure provides a common two-dimensional basis for visual comparison.

This is a post-challenge cleaned version of the original experiment notebook. Duplicate plotting cells, local paths, and intermediate layout experiments were removed while retaining the analysis procedure used for the paper.


In [ ]:
import gc
import inspect
import json
import math
import random
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import albumentations as A
import cv2
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from albumentations.pytorch import ToTensorV2
from IPython.display import display
from matplotlib.colors import BoundaryNorm
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")


## 1. Plotting and experiment configuration

In [ ]:
# Embed TrueType fonts in vector outputs.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# Paper-oriented font sizes.
mpl.rcParams["font.size"] = 8
mpl.rcParams["axes.titlesize"] = 8
mpl.rcParams["axes.labelsize"] = 8
mpl.rcParams["xtick.labelsize"] = 8
mpl.rcParams["ytick.labelsize"] = 8
mpl.rcParams["legend.fontsize"] = 8


@dataclass
class CFG:
    seed: int = 42
    num_classes: int = 10

    data_dir_name: str = "Data"
    train_csv_name: str = "training.csv"

    model_name: str = (
        "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"
    )
    image_size: int = 384
    embedding_dim: int = 512

    n_splits: int = 5
    valid_bs: int = 32
    num_workers: int = 0

    dropout: float = 0.2
    arc_s: float = 30.0
    arc_m: float = 0.30

    # Full precision is used to reproduce embedding geometry consistently.
    use_amp: bool = False
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()

ANALYSIS_FOLD = 0
TSNE_PERPLEXITY = 50
TSNE_ITERATIONS = 1500

# Fold 0 is also used for the Procrustes/shared-PCA visualization.
PROCRUSTES_FOLD = 0
REFERENCE_MODEL_NAME = "arcbase_p4"


## 2. Repository paths and reproducibility

In [ ]:
def find_repository_root(start_path):
    current = Path(start_path).resolve()

    while True:
        has_readme = (current / "README.md").exists()
        has_data_dir = (current / cfg.data_dir_name).exists()

        if has_readme and has_data_dir:
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Repository root could not be located. "
        "Run this notebook inside the cloned repository."
    )


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


REPO_ROOT = find_repository_root(Path.cwd())
DATA_ROOT = REPO_ROOT / cfg.data_dir_name
TRAIN_CSV = DATA_ROOT / cfg.train_csv_name

ARTIFACT_ROOT = REPO_ROOT / "ensemble_artifacts"
DIAGNOSTIC_CHECKPOINT_DIR = ARTIFACT_ROOT / "diagnostics"
ANALYSIS_CACHE_DIR = (
    DIAGNOSTIC_CHECKPOINT_DIR / "representation_analysis"
)
RESULT_DIR = REPO_ROOT / "results" / "diagnostics"

ANALYSIS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg.seed)
analysis_start_time = time.time()

print("Repository root :", REPO_ROOT)
print("Training CSV    :", TRAIN_CSV)
print("Cache directory :", ANALYSIS_CACHE_DIR)
print("Result directory:", RESULT_DIR)
print("Device          :", cfg.device)


## 3. Training data and deterministic validation preprocessing

In [ ]:
if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Training CSV not found: {TRAIN_CSV}")


def make_abs_path(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return str(path)
    return str(REPO_ROOT / path)


train_df = pd.read_csv(TRAIN_CSV)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError(
        "training.csv must contain either 'y' or 'TARGET'."
    )

required_columns = {"path", "label"}
missing_columns = required_columns - set(train_df.columns)
if missing_columns:
    raise ValueError(
        f"Missing training columns: {sorted(missing_columns)}"
    )

train_df["filepath"] = train_df["path"].apply(make_abs_path)

print("train_df:", train_df.shape)
print(train_df["label"].value_counts().sort_index())


In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


# =========================================================
# Dataset
# =========================================================
class BaseImageDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None,
        is_test=False,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        image = cv2.imread(row["filepath"])

        if image is None:
            raise FileNotFoundError(row["filepath"])

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB,
        )

        if self.transform is not None:
            image = self.transform(
                image=image
            )["image"]

        if self.is_test:
            return image, int(row["ID"])

        return image, int(row["label"])


## 4. ConvNeXtV2 model and inference-head fusion

In [ ]:
class ArcMarginProduct(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        s=30.0,
        m=0.30,
    ):
        super().__init__()

        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
            )
        )

        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        cosine = F.linear(
            F.normalize(embeddings),
            F.normalize(self.weight),
        ).clamp(-1.0, 1.0)

        if labels is None:
            return cosine * self.s

        sine = torch.sqrt(
            torch.clamp(
                1.0 - cosine.pow(2),
                min=1e-7,
            )
        )

        phi = (
            cosine * self.cos_m
            - sine * self.sin_m
        )

        phi = torch.where(
            cosine > self.th,
            phi,
            cosine - self.mm,
        )

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(
            1,
            labels.view(-1, 1).long(),
            1.0,
        )

        logits = (
            one_hot * phi
            + (1.0 - one_hot) * cosine
        )

        return logits * self.s


class ArcFaceModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes,
        embedding_dim=512,
        arc_s=30.0,
        arc_m=0.30,
        dropout=0.2,
        pretrained=True,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        backbone_out = self.backbone.num_features

        self.neck = nn.Sequential(
            nn.Linear(
                backbone_out,
                embedding_dim,
            ),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU(),
        )

        self.dropout = nn.Dropout(dropout)

        self.ce_head = nn.Linear(
            embedding_dim,
            num_classes,
        )

        self.arc_head = ArcMarginProduct(
            embedding_dim,
            num_classes,
            s=arc_s,
            m=arc_m,
        )

    def forward(self, images, labels=None):
        features = self.backbone(images)
        embeddings = self.neck(features)
        embeddings = self.dropout(embeddings)

        ce_logits = self.ce_head(embeddings)

        arc_logits = self.arc_head(
            embeddings,
            labels,
        )

        return ce_logits, arc_logits, embeddings


# =========================================================
# Inference fusion
# =========================================================
def fuse_logits(
    ce_logits,
    arc_logits,
    infer_ce,
    infer_arc,
):
    total = infer_ce + infer_arc

    if total <= 0:
        raise ValueError(
            "infer_ce + infer_arc must be greater than zero."
        )

    infer_ce = infer_ce / total
    infer_arc = infer_arc / total

    return (
        infer_ce * ce_logits
        + infer_arc * arc_logits
    )


## 5. Reconstruct the original five validation folds

In [ ]:
skf = StratifiedKFold(
    n_splits=cfg.n_splits,
    shuffle=True,
    random_state=cfg.seed,
)

splits = list(
    skf.split(
        train_df,
        train_df["label"],
    )
)

print("Prepared folds:", len(splits))
print("Device:", cfg.device)


## 6. Analysis models and checkpoint locations

The diagnostic CE-only, ArcFace-only, and CE+ArcFace (20:80) checkpoints are expected under `ensemble_artifacts/diagnostics/`. The official CE+ArcFace (60:40) checkpoint remains under `ensemble_artifacts/`.


In [ ]:
ANALYSIS_MODELS = [
    {
        "name": "ce_only_p4",
        "display_name": "CE-only",
        "checkpoint_source": (
            "post-challenge patience-4 diagnostic run"
        ),
        "checkpoint_pattern": str(
            DIAGNOSTIC_CHECKPOINT_DIR
            / "arcface_base_ce_only_best_fold{fold}.pth"
        ),
        "train_ce": 1.0,
        "train_arc": 0.0,
        "infer_ce": 1.0,
        "infer_arc": 0.0,
    },
    {
        "name": "arc_only_p4",
        "display_name": "ArcFace-only",
        "checkpoint_source": (
            "post-challenge patience-4 diagnostic run"
        ),
        "checkpoint_pattern": str(
            DIAGNOSTIC_CHECKPOINT_DIR
            / "arcface_base_arcface_only_best_fold{fold}.pth"
        ),
        "train_ce": 0.0,
        "train_arc": 1.0,
        "infer_ce": 0.0,
        "infer_arc": 1.0,
    },
    {
        "name": "arcbase_p4",
        "display_name": "CE+ArcFace (60:40)",
        "checkpoint_source": (
            "official challenge arcface-base branch, patience 4"
        ),
        "checkpoint_pattern": str(
            ARTIFACT_ROOT
            / "arcface_base_best_fold{fold}.pth"
        ),
        "train_ce": 0.6,
        "train_arc": 0.4,
        "infer_ce": 0.5,
        "infer_arc": 0.5,
    },
    {
        "name": "ce20arc80_p4",
        "display_name": "CE+ArcFace (20:80)",
        "checkpoint_source": (
            "post-challenge patience-4 diagnostic run"
        ),
        "checkpoint_pattern": str(
            DIAGNOSTIC_CHECKPOINT_DIR
            / "arcface_base_ce20arc80_best_fold{fold}.pth"
        ),
        "train_ce": 0.2,
        "train_arc": 0.8,
        "infer_ce": 0.5,
        "infer_arc": 0.5,
    },
]


def resolve_analysis_checkpoint_path(analysis_setting, fold):
    return Path(
        analysis_setting["checkpoint_pattern"].format(
            fold=fold
        )
    )

def repository_relative_path(path):
    """
    Convert a local absolute path into a repository-relative,
    platform-independent path for public result files.
    """
    path = Path(path).resolve()
    repository_root = REPO_ROOT.resolve()

    try:
        relative_path = path.relative_to(
            repository_root
        )
    except ValueError as error:
        raise ValueError(
            "The checkpoint is outside the repository root: "
            f"{path}"
        ) from error

    return relative_path.as_posix()


## 7. Checkpoint loading and representation-analysis utilities

In [ ]:
def extract_state_dict(checkpoint):
    """
    Support both:
      1. a directly saved model.state_dict()
      2. a dictionary containing model_state_dict or state_dict
    """
    if not isinstance(checkpoint, dict):
        raise TypeError(
            "The loaded checkpoint is not a dictionary."
        )

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        # The original arcface_base checkpoint was saved
        # directly as model.state_dict().
        state_dict = checkpoint

    # Support checkpoints saved from DataParallel.
    if any(key.startswith("module.") for key in state_dict):
        state_dict = {
            key.removeprefix("module."): value
            for key, value in state_dict.items()
        }

    return state_dict


def load_analysis_checkpoint(analysis_setting, fold):
    """
    Load either a diagnostic checkpoint or the archived official
    arcface_base checkpoint.
    """
    checkpoint_path = resolve_analysis_checkpoint_path(
        analysis_setting,
        fold,
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=cfg.device,
    )

    state_dict = extract_state_dict(checkpoint)

    model = ArcFaceModel(
        cfg.model_name,
        cfg.num_classes,
        embedding_dim=cfg.embedding_dim,
        arc_s=cfg.arc_s,
        arc_m=cfg.arc_m,
        dropout=cfg.dropout,
        pretrained=False,
    ).to(cfg.device)

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    return model, checkpoint_path, checkpoint


@torch.no_grad()
def extract_normalized_embeddings(model, dataframe, infer_ce, infer_arc):
    dataset = BaseImageDataset(
        dataframe,
        transform=get_valid_transforms(cfg.image_size),
        is_test=False,
    )
    loader = DataLoader(
        dataset,
        batch_size=cfg.valid_bs,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=False,
    )

    embedding_batches = []
    probability_batches = []
    label_batches = []

    for images, labels in loader:
        images = images.to(cfg.device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_cosine_logits, embeddings = model(images, labels=None)
            logits = fuse_logits(
                ce_logits,
                arc_cosine_logits,
                infer_ce,
                infer_arc,
            )
            probabilities = torch.softmax(logits, dim=1)

        embedding_batches.append(
            F.normalize(embeddings.float(), p=2, dim=1).cpu().numpy()
        )
        probability_batches.append(probabilities.cpu().numpy())
        label_batches.append(labels.numpy())

    embeddings = np.concatenate(embedding_batches, axis=0).astype(np.float32)
    probabilities = np.concatenate(probability_batches, axis=0).astype(np.float32)
    labels = np.concatenate(label_batches, axis=0).astype(np.int64)
    predictions = probabilities.argmax(axis=1).astype(np.int64)

    del dataset, loader
    gc.collect()
    return embeddings, probabilities, labels, predictions


def normalize_rows(values):
    values = np.asarray(values, dtype=np.float64)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.clip(norms, 1e-12, None)


def representation_metrics(embeddings, labels, predictions):
    embeddings = normalize_rows(embeddings)
    classes = np.sort(np.unique(labels))

    centroids = []
    intra_distances = []
    for class_id in classes:
        class_embeddings = embeddings[labels == class_id]
        centroid = normalize_rows(class_embeddings.mean(axis=0, keepdims=True))[0]
        centroids.append(centroid)
        intra_distances.extend((1.0 - class_embeddings @ centroid).tolist())

    centroids = np.stack(centroids, axis=0)
    similarities = np.clip(centroids @ centroids.T, -1.0, 1.0)
    angles = np.degrees(np.arccos(similarities))
    upper = np.triu(np.ones_like(angles, dtype=bool), k=1)
    pair_angles = angles[upper]

    nearest_centroid_predictions = classes[
        np.argmax(embeddings @ centroids.T, axis=1)
    ]

    class_6_7_angle = np.nan
    if 6 in classes and 7 in classes:
        pos6 = int(np.where(classes == 6)[0][0])
        pos7 = int(np.where(classes == 7)[0][0])
        class_6_7_angle = float(angles[pos6, pos7])

    try:
        cosine_silhouette = float(
            silhouette_score(embeddings, labels, metric="cosine")
        )
    except Exception as error:
        print("Silhouette score failed:", error)
        cosine_silhouette = np.nan

    return {
        "classification_accuracy": float(accuracy_score(labels, predictions)),
        "cosine_silhouette": cosine_silhouette,
        "mean_intra_class_cosine_distance": float(np.mean(intra_distances)),
        "mean_centroid_angle_deg": float(np.mean(pair_angles)),
        "minimum_centroid_angle_deg": float(np.min(pair_angles)),
        "nearest_centroid_accuracy": float(
            accuracy_score(labels, nearest_centroid_predictions)
        ),
        "class_6_7_centroid_angle_deg": class_6_7_angle,
    }


def fit_tsne(embeddings):
    effective_perplexity = min(
        TSNE_PERPLEXITY,
        max(5, (len(embeddings) - 1) // 3),
    )
    kwargs = {
        "n_components": 2,
        "perplexity": effective_perplexity,
        "learning_rate": "auto",
        "init": "pca",
        "random_state": cfg.seed,
        "metric": "euclidean",
        "method": "barnes_hut",
        "angle": 0.5,
        "verbose": 1,
    }
    parameters = inspect.signature(TSNE).parameters
    if "max_iter" in parameters:
        kwargs["max_iter"] = TSNE_ITERATIONS
    else:
        kwargs["n_iter"] = TSNE_ITERATIONS
    if "n_jobs" in parameters:
        kwargs["n_jobs"] = -1
    model = TSNE(**kwargs)
    return model.fit_transform(embeddings), model


## 8. Verify all required checkpoints

In [ ]:
missing_checkpoints = []

for analysis_setting in ANALYSIS_MODELS:
    print(f"\n[{analysis_setting['display_name']}]")

    for fold in range(cfg.n_splits):
        checkpoint_path = resolve_analysis_checkpoint_path(
            analysis_setting,
            fold,
        )

        if checkpoint_path.exists():
            print(
                f"  fold {fold}: {checkpoint_path} "
                f"({checkpoint_path.stat().st_size:,} bytes)"
            )
        else:
            print(f"  fold {fold}: MISSING -> {checkpoint_path}")
            missing_checkpoints.append(str(checkpoint_path))

if missing_checkpoints:
    raise FileNotFoundError(
        "The following analysis checkpoints were not found:\n"
        + "\n".join(missing_checkpoints)
    )


## 9. Extract held-out embeddings and compute original-space metrics

For every fold and objective, this section:

- loads the saved best checkpoint;
- extracts L2-normalized 512-dimensional held-out embeddings;
- predicts with the original inference-head fusion;
- computes representation metrics in the original embedding space;
- fits an independent t-SNE only for the selected visualization fold.

Intermediate arrays are written to the ignored artifact cache. Publication tables and figures are written to `results/diagnostics/`.


In [ ]:
metric_rows = []
tsne_panels = []

for analysis_setting in ANALYSIS_MODELS:
    train_name = analysis_setting["name"]
    display_name = analysis_setting["display_name"]
    infer_ce = analysis_setting["infer_ce"]
    infer_arc = analysis_setting["infer_arc"]

    objective_analysis_dir = ANALYSIS_CACHE_DIR / train_name
    objective_analysis_dir.mkdir(parents=True, exist_ok=True)

    for fold, (_, va_idx) in enumerate(splits):
        val_df = train_df.iloc[va_idx].reset_index(drop=True)
        model, checkpoint_path, checkpoint = load_analysis_checkpoint(
            analysis_setting,
            fold,
        )

        embeddings, probabilities, labels, predictions = (
            extract_normalized_embeddings(
                model,
                val_df,
                infer_ce=infer_ce,
                infer_arc=infer_arc,
            )
        )

        np.save(
            objective_analysis_dir / f"fold{fold}_validation_embeddings_l2.npy",
            embeddings,
        )
        np.save(
            objective_analysis_dir / f"fold{fold}_validation_probabilities.npy",
            probabilities,
        )
        np.save(
            objective_analysis_dir / f"fold{fold}_validation_labels.npy",
            labels,
        )
        np.save(
            objective_analysis_dir / f"fold{fold}_validation_predictions.npy",
            predictions,
        )
        np.save(
            objective_analysis_dir / f"fold{fold}_validation_original_indices.npy",
            np.asarray(va_idx, dtype=np.int64),
        )

        metrics = representation_metrics(embeddings, labels, predictions)
        metric_rows.append({
            "train_name": train_name,
            "display_name": display_name,
            "fold": fold,
            "train_ce": analysis_setting["train_ce"],
            "train_arc": analysis_setting["train_arc"],
            "checkpoint_source": analysis_setting[
                "checkpoint_source"
            ],
            "infer_ce": infer_ce,
            "infer_arc": infer_arc,
            "checkpoint_path": repository_relative_path(
                checkpoint_path
            ),
            **metrics,
        })

        if fold == ANALYSIS_FOLD:
            coordinates, tsne_model = fit_tsne(embeddings)
            coordinate_df = pd.DataFrame({
                "original_index": np.asarray(va_idx, dtype=np.int64),
                "tsne_x": coordinates[:, 0],
                "tsne_y": coordinates[:, 1],
                "true_label": labels,
                "predicted_label": predictions,
                "correct": predictions == labels,
            })
            coordinate_path = (
                objective_analysis_dir
                / f"fold{fold}_tsne_coordinates.csv"
            )
            coordinate_df.to_csv(coordinate_path, index=False)
            tsne_panels.append({
                "train_name": train_name,
                "display_name": display_name,
                "coordinates": coordinates,
                "labels": labels,
                "predictions": predictions,
                "kl_divergence": float(tsne_model.kl_divergence_),
            })

        del model, embeddings, probabilities
        gc.collect()
        torch.cuda.empty_cache()



In [ ]:
metrics_df = pd.DataFrame(metric_rows)

metric_columns = [
    "classification_accuracy",
    "cosine_silhouette",
    "mean_intra_class_cosine_distance",
    "mean_centroid_angle_deg",
    "minimum_centroid_angle_deg",
    "nearest_centroid_accuracy",
    "class_6_7_centroid_angle_deg",
]

# Set the model order to match Table 1 in the paper.
paper_model_order = [
    "ce_only_p4",
    "arc_only_p4",
    "arcbase_p4",
    "ce20arc80_p4",
]

model_order_map = {
    train_name: order
    for order, train_name in enumerate(paper_model_order)
}

# Check that every model has a defined display order.
unknown_train_names = sorted(
    set(metrics_df["train_name"])
    - set(paper_model_order)
)

if unknown_train_names:
    raise ValueError(
        "The following train_name values are missing from "
        f"paper_model_order: {unknown_train_names}"
    )

# Add a temporary sort key.
metrics_df["_paper_order"] = (
    metrics_df["train_name"]
    .map(model_order_map)
)

# Sort fold-level results in the same model order as the paper.
sort_columns = ["_paper_order"]

if "fold" in metrics_df.columns:
    sort_columns.append("fold")

metrics_df = (
    metrics_df
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

# Calculate the fold mean for each model.
summary_mean = (
    metrics_df
    .groupby(
        ["train_name", "display_name"],
        as_index=False,
        sort=False,
    )[metric_columns]
    .mean()
)

# Calculate the fold standard deviation for each model.
summary_std = (
    metrics_df
    .groupby(
        ["train_name", "display_name"],
        as_index=False,
        sort=False,
    )[metric_columns]
    .std(ddof=1)
)

# Apply the paper order explicitly to both summary tables.
for summary_df in [summary_mean, summary_std]:
    summary_df["_paper_order"] = (
        summary_df["train_name"]
        .map(model_order_map)
    )

    summary_df.sort_values(
        "_paper_order",
        inplace=True,
    )

    summary_df.drop(
        columns="_paper_order",
        inplace=True,
    )

    summary_df.reset_index(
        drop=True,
        inplace=True,
    )

# Remove the temporary sort key from the fold-level table.
metrics_df.drop(
    columns="_paper_order",
    inplace=True,
)

# Save the fold-level metrics and the paper-facing summary tables.
metrics_df.to_csv(
    RESULT_DIR / "representation_metrics_by_fold.csv",
    index=False,
)

summary_mean.to_csv(
    RESULT_DIR / "table1_representation_metrics.csv",
    index=False,
)

summary_std.to_csv(
    RESULT_DIR / "representation_metrics_std.csv",
    index=False,
)

print("Representation metrics (fold mean):")
display(summary_mean)

print("Representation metrics (fold standard deviation):")
display(summary_std)

## 10. Independent t-SNE projections

Each model is projected with a separately fitted t-SNE model. Therefore, absolute axes, rotations, reflections, and distances between panels must not be compared directly. These panels are supplementary visualizations rather than the primary cross-model geometric evidence.


In [ ]:
cmap = plt.get_cmap(
    "tab10",
    cfg.num_classes,
)

norm = BoundaryNorm(
    np.arange(
        -0.5,
        cfg.num_classes + 0.5,
        1.0,
    ),
    cmap.N,
)

panel_count = len(tsne_panels)

figure, axes = plt.subplots(
    1,
    panel_count,
    figsize=(1.75 * panel_count, 2.15),
    constrained_layout=False,
)

if panel_count == 1:
    axes = [axes]

for panel_index, (axis, panel) in enumerate(
    zip(axes, tsne_panels)
):
    coordinates = panel["coordinates"]
    labels = panel["labels"]
    predictions = panel["predictions"]

    correct = predictions == labels
    errors = ~correct

    # Correctly classified samples.
    # rasterized=False keeps every marker as vector graphics.
    axis.scatter(
        coordinates[correct, 0],
        coordinates[correct, 1],
        c=labels[correct],
        cmap=cmap,
        norm=norm,
        s=5,
        alpha=0.65,
        linewidths=0,
        rasterized=False,
    )

    # Misclassified samples are marked with crosses.
    if errors.any():
        axis.scatter(
            coordinates[errors, 0],
            coordinates[errors, 1],
            c=labels[errors],
            cmap=cmap,
            norm=norm,
            s=18,
            marker="x",
            alpha=0.95,
            linewidths=0.8,
            rasterized=False,
        )

    panel_accuracy = accuracy_score(
        labels,
        predictions,
    )

    axis.set_title(
        f"{panel['display_name']}\n"
        f"acc. {panel_accuracy:.4f}",
        pad=2,
    )

    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.grid(alpha=0.15, linewidth=0.4)

    # Keep y tick labels only on the first panel to save space.
    if panel_index > 0:
        axis.tick_params(labelleft=False)

# Shared axis labels are printed only once for the complete figure.
figure.supxlabel(
    "t-SNE dimension 1",
    fontsize=9,
    y=0.15,
)

figure.supylabel(
    "t-SNE dimension 2",
    fontsize=9,
    rotation=90,
    x=0.006,
    y=0.70,
    va="center",
    linespacing=0.9,
)

# =========================================================
# Detached legend below the complete figure
# =========================================================
class_legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor=cmap(norm(class_id)),
        markeredgecolor="none",
        markersize=4.5,
        label=str(class_id),
    )
    for class_id in range(cfg.num_classes)
]

error_legend_handle = Line2D(
    [0],
    [0],
    marker="x",
    linestyle="None",
    color="black",
    markersize=5,
    markeredgewidth=0.9,
    label="Misclassified",
)

figure.legend(
    handles=(
        class_legend_handles
        + [error_legend_handle]
    ),
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=11,
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="grey",

    title="Source class",
    fontsize=9,
    title_fontsize=9,

    handletextpad=0.1,
    columnspacing=0.55,
    borderpad=0.2,
    borderaxespad=0.0,  
)

# =========================================================
# Layout
# =========================================================
figure.subplots_adjust(
    top=0.94,
    bottom=0.34,
    left=0.075,
    right=0.99,
    wspace=0.24,
)

comparison_png = (
    RESULT_DIR
    / "supplementary_tsne_fold0.png"
)

comparison_pdf = (
    RESULT_DIR
    / "supplementary_tsne_fold0.pdf"
)

# Publication version: genuine vector PDF.
figure.savefig(
    comparison_pdf,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.04,
)

# Screen-check version only.
figure.savefig(
    comparison_png,
    format="png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.4,
)

plt.show()
plt.close(figure)

print("Saved vector figure:", comparison_pdf)
print("Saved preview figure:", comparison_png)


## 11. Orthogonal Procrustes alignment and shared PCA

To place all four embedding spaces in one common visualization:

1. fold-0 L2-normalized embeddings are loaded in identical sample order;
2. each space is aligned to CE+ArcFace (60:40) using an orthogonal rotation/reflection only;
3. no translation or scaling is applied;
4. one PCA model is fitted once to the concatenation of all aligned embeddings;
5. every model is transformed with that same PCA basis.

The quantitative metrics above remain calculated in the original 512-dimensional spaces.


In [ ]:
COMMON_PCA_CACHE_DIR = (
    ANALYSIS_CACHE_DIR / "procrustes_shared_pca"
)
COMMON_PCA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    {
        "name": "ce_only_p4",
        "display_name": "CE-only",
    },
    {
        "name": "arc_only_p4",
        "display_name": "ArcFace-only",
    },
    {
        "name": "arcbase_p4",
        "display_name": "CE+ArcFace (60:40)",
    },
    {
        "name": "ce20arc80_p4",
        "display_name": "CE+ArcFace (20:80)",
    },
]


def l2_normalize_rows(values, eps=1e-12):
    values = np.asarray(values, dtype=np.float64)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.clip(norms, eps, None)


def load_embedding_bundle(model_spec, fold):
    model_dir = ANALYSIS_CACHE_DIR / model_spec["name"]
    prefix = f"fold{fold}_validation"

    paths = {
        "embeddings": model_dir / f"{prefix}_embeddings_l2.npy",
        "labels": model_dir / f"{prefix}_labels.npy",
        "predictions": model_dir / f"{prefix}_predictions.npy",
        "original_indices": (
            model_dir / f"{prefix}_original_indices.npy"
        ),
    }

    missing = [
        str(path)
        for path in paths.values()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "The following representation files were not found:\n"
            + "\n".join(missing)
        )

    return {
        "name": model_spec["name"],
        "display_name": model_spec["display_name"],
        "embeddings": l2_normalize_rows(
            np.load(paths["embeddings"])
        ),
        "labels": np.load(paths["labels"]).astype(np.int64),
        "predictions": np.load(
            paths["predictions"]
        ).astype(np.int64),
        "original_indices": np.load(
            paths["original_indices"]
        ).astype(np.int64),
    }


def orthogonal_procrustes_align(
    source_embeddings,
    reference_embeddings,
):
    source_embeddings = np.asarray(
        source_embeddings,
        dtype=np.float64,
    )
    reference_embeddings = np.asarray(
        reference_embeddings,
        dtype=np.float64,
    )

    if source_embeddings.shape != reference_embeddings.shape:
        raise ValueError(
            "Source and reference embeddings must have "
            "identical shapes."
        )

    cross_covariance = (
        source_embeddings.T @ reference_embeddings
    )
    left_vectors, singular_values, right_vectors_t = (
        np.linalg.svd(
            cross_covariance,
            full_matrices=False,
        )
    )
    rotation_matrix = left_vectors @ right_vectors_t
    aligned_embeddings = source_embeddings @ rotation_matrix

    return (
        aligned_embeddings,
        rotation_matrix,
        singular_values,
    )


def alignment_diagnostics(
    original_embeddings,
    aligned_embeddings,
    reference_embeddings,
    rotation_matrix,
):
    paired_cosine_before = np.mean(
        np.sum(
            original_embeddings * reference_embeddings,
            axis=1,
        )
    )
    paired_cosine_after = np.mean(
        np.sum(
            aligned_embeddings * reference_embeddings,
            axis=1,
        )
    )
    rms_alignment_error = np.sqrt(
        np.mean(
            np.sum(
                (
                    aligned_embeddings
                    - reference_embeddings
                ) ** 2,
                axis=1,
            )
        )
    )
    orthogonality_error = np.linalg.norm(
        (
            rotation_matrix.T @ rotation_matrix
        )
        - np.eye(rotation_matrix.shape[0]),
        ord="fro",
    )

    sample_count = min(300, len(original_embeddings))
    sample_indices = np.linspace(
        0,
        len(original_embeddings) - 1,
        sample_count,
        dtype=int,
    )

    original_subset = original_embeddings[sample_indices]
    aligned_subset = aligned_embeddings[sample_indices]
    original_gram = original_subset @ original_subset.T
    aligned_gram = aligned_subset @ aligned_subset.T

    max_cosine_geometry_change = np.max(
        np.abs(original_gram - aligned_gram)
    )
    determinant_sign, determinant_log_abs = np.linalg.slogdet(
        rotation_matrix
    )

    return {
        "paired_cosine_before": float(
            paired_cosine_before
        ),
        "paired_cosine_after": float(
            paired_cosine_after
        ),
        "rms_alignment_error": float(
            rms_alignment_error
        ),
        "rotation_determinant_sign": float(
            determinant_sign
        ),
        "rotation_log_abs_determinant": float(
            determinant_log_abs
        ),
        "orthogonality_error": float(
            orthogonality_error
        ),
        "max_cosine_geometry_change": float(
            max_cosine_geometry_change
        ),
    }


In [ ]:
bundles = {
    spec["name"]: load_embedding_bundle(
        spec,
        PROCRUSTES_FOLD,
    )
    for spec in MODEL_SPECS
}

if REFERENCE_MODEL_NAME not in bundles:
    raise KeyError(
        f"Reference model was not found: "
        f"{REFERENCE_MODEL_NAME}"
    )

reference_bundle = bundles[
    REFERENCE_MODEL_NAME
]

reference_indices = reference_bundle[
    "original_indices"
]

reference_labels = reference_bundle[
    "labels"
]


# =========================================================
# Verify identical samples and ordering
# =========================================================
for model_name, bundle in bundles.items():
    if not np.array_equal(
        bundle["original_indices"],
        reference_indices,
    ):
        raise ValueError(
            "Original validation indices differ between "
            f"{model_name} and {REFERENCE_MODEL_NAME}."
        )

    if not np.array_equal(
        bundle["labels"],
        reference_labels,
    ):
        raise ValueError(
            "Validation labels differ between "
            f"{model_name} and {REFERENCE_MODEL_NAME}."
        )

print(
    "Verified identical validation samples:",
    len(reference_indices),
)

print(
    "Reference model:",
    reference_bundle["display_name"],
)


In [ ]:
reference_embeddings = reference_bundle[
    "embeddings"
]

aligned_embeddings = {}
rotation_matrices = {}
diagnostic_rows = []

for model_spec in MODEL_SPECS:
    model_name = model_spec["name"]
    bundle = bundles[model_name]

    original_embeddings = bundle[
        "embeddings"
    ]

    if model_name == REFERENCE_MODEL_NAME:
        aligned = original_embeddings.copy()

        rotation_matrix = np.eye(
            original_embeddings.shape[1],
            dtype=np.float64,
        )

        singular_values = np.ones(
            original_embeddings.shape[1],
            dtype=np.float64,
        )

    else:
        (
            aligned,
            rotation_matrix,
            singular_values,
        ) = orthogonal_procrustes_align(
            source_embeddings=original_embeddings,
            reference_embeddings=reference_embeddings,
        )

    # Orthogonal transformation should preserve norms.
    # Normalize again only to remove numerical round-off.
    aligned = l2_normalize_rows(
        aligned
    )

    aligned_embeddings[model_name] = aligned
    rotation_matrices[model_name] = rotation_matrix

    diagnostics = alignment_diagnostics(
        original_embeddings=original_embeddings,
        aligned_embeddings=aligned,
        reference_embeddings=reference_embeddings,
        rotation_matrix=rotation_matrix,
    )

    diagnostic_rows.append({
        "model_name": model_name,
        "display_name": bundle["display_name"],
        "reference_model": REFERENCE_MODEL_NAME,
        "is_reference": (
            model_name == REFERENCE_MODEL_NAME
        ),
        "sum_singular_values": float(
            singular_values.sum()
        ),
        **diagnostics,
    })

    np.save(
        COMMON_PCA_CACHE_DIR
        / (
            f"{model_name}_fold"
            f"{PROCRUSTES_FOLD}_aligned_embeddings.npy"
        ),
        aligned.astype(np.float32),
    )

    np.save(
        COMMON_PCA_CACHE_DIR
        / (
            f"{model_name}_fold"
            f"{PROCRUSTES_FOLD}_rotation_matrix.npy"
        ),
        rotation_matrix.astype(np.float32),
    )


alignment_diagnostics_df = pd.DataFrame(
    diagnostic_rows
)

alignment_diagnostics_df.to_csv(
    RESULT_DIR / "procrustes_alignment_diagnostics.csv",
    index=False,
)

print("\nProcrustes alignment diagnostics:")
display(alignment_diagnostics_df)


In [ ]:
all_aligned_embeddings = np.concatenate(
    [
        aligned_embeddings[
            model_spec["name"]
        ]
        for model_spec in MODEL_SPECS
    ],
    axis=0,
)

shared_pca = PCA(
    n_components=2,
    svd_solver="full",
)

shared_pca.fit(
    all_aligned_embeddings
)

explained_variance_ratio = (
    shared_pca.explained_variance_ratio_
)

print(
    "\nShared PCA explained variance ratio:",
    explained_variance_ratio,
)

print(
    "Shared PCA cumulative explained variance:",
    explained_variance_ratio.sum(),
)


# =========================================================
# Transform each model using the same PCA basis
# =========================================================
common_pca_panels = []

for model_spec in MODEL_SPECS:
    model_name = model_spec["name"]
    bundle = bundles[model_name]

    coordinates = shared_pca.transform(
        aligned_embeddings[model_name]
    )

    coordinate_df = pd.DataFrame({
        "original_index": bundle[
            "original_indices"
        ],
        "pca_x": coordinates[:, 0],
        "pca_y": coordinates[:, 1],
        "true_label": bundle["labels"],
        "predicted_label": bundle[
            "predictions"
        ],
        "correct": (
            bundle["predictions"]
            == bundle["labels"]
        ),
    })

    coordinate_df.to_csv(
        COMMON_PCA_CACHE_DIR
        / (
            f"{model_name}_fold"
            f"{PROCRUSTES_FOLD}_common_pca_coordinates.csv"
        ),
        index=False,
    )

    common_pca_panels.append({
        "name": model_name,
        "display_name": bundle["display_name"],
        "coordinates": coordinates,
        "labels": bundle["labels"],
        "predictions": bundle["predictions"],
    })


# =========================================================
# Use identical axis limits across all four panels
# =========================================================
all_coordinates = np.concatenate(
    [
        panel["coordinates"]
        for panel in common_pca_panels
    ],
    axis=0,
)

x_min = float(
    all_coordinates[:, 0].min()
)

x_max = float(
    all_coordinates[:, 0].max()
)

y_min = float(
    all_coordinates[:, 1].min()
)

y_max = float(
    all_coordinates[:, 1].max()
)

x_padding = max(
    (x_max - x_min) * 0.05,
    1e-3,
)

y_padding = max(
    (y_max - y_min) * 0.05,
    1e-3,
)

shared_x_limits = (
    x_min - x_padding,
    x_max + x_padding,
)

shared_y_limits = (
    y_min - y_padding,
    y_max + y_padding,
)


## 12. Publication-oriented shared-PCA figure

In [ ]:
# Draw common-PCA comparison
# =========================================================
# This is the publication-oriented version of Fig. 2.
# The PDF is generated directly by Matplotlib so that scatter
# markers, axes, labels, and legend remain vector graphics.

cmap = plt.get_cmap(
    "tab10",
    10,
)

norm = BoundaryNorm(
    np.arange(
        -0.5,
        10.5,
        1.0,
    ),
    cmap.N,
)

panel_count = len(common_pca_panels)

figure, axes = plt.subplots(
    1,
    panel_count,
    figsize=(1.75 * panel_count, 2.15),
    sharex=True,
    sharey=True,
    constrained_layout=False,
)

if panel_count == 1:
    axes = [axes]

for panel_index, (axis, panel) in enumerate(
    zip(axes, common_pca_panels)
):
    coordinates = panel[
        "coordinates"
    ]

    labels = panel["labels"]

    predictions = panel[
        "predictions"
    ]

    correct = (
        predictions == labels
    )

    errors = ~correct

    # Keep all points as vector objects in the PDF.
    axis.scatter(
        coordinates[correct, 0],
        coordinates[correct, 1],
        c=labels[correct],
        cmap=cmap,
        norm=norm,
        s=5,
        alpha=0.65,
        linewidths=0,
        rasterized=False,
    )

    if errors.any():
        axis.scatter(
            coordinates[errors, 0],
            coordinates[errors, 1],
            c=labels[errors],
            cmap=cmap,
            norm=norm,
            s=18,
            marker="x",
            alpha=0.95,
            linewidths=0.8,
            rasterized=False,
        )

    panel_accuracy = accuracy_score(
        labels,
        predictions,
    )

    axis.set_title(
        f"{panel['display_name']}\n"
        f"acc. {panel_accuracy:.4f}",
        pad=2,
    )

    # Axis labels are shared across the four panels.
    axis.set_xlabel("")
    axis.set_ylabel("")

    axis.set_xlim(
        shared_x_limits
    )

    axis.set_ylim(
        shared_y_limits
    )

    # Make one unit on the x-axis equal to one unit on the y-axis.
    axis.set_aspect(
        "equal",
        adjustable="box",
    )

    axis.grid(
        alpha=0.15,
        linewidth=0.4,
    )

    # All panels use common limits, so repeated y tick labels
    # are unnecessary.
    if panel_index > 0:
        axis.tick_params(labelleft=False)

# Print the shared labels only once to improve readability at LNCS width.
# =========================================================
# Common axis labels
# =========================================================
figure.supxlabel(
    "Shared PCA component 1 "
    f"({explained_variance_ratio[0] * 100:.1f}%)",
    fontsize=9,
    y=0.15,
)

figure.supylabel(
    "Shared PCA\n"
    f"component 2 ({explained_variance_ratio[1] * 100:.1f}%)",
    fontsize=9,
    rotation=90,
    x=-0.02,
    y=0.70,
    va="center",
    linespacing=0.9,
)

# =========================================================
# Detached legend below the complete figure
# =========================================================
class_legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor=cmap(norm(class_id)),
        markeredgecolor="none",
        markersize=4.5,
        label=str(class_id),
    )
    for class_id in range(10)
]

error_legend_handle = Line2D(
    [0],
    [0],
    marker="x",
    linestyle="None",
    color="black",
    markersize=5,
    markeredgewidth=0.9,
    label="Misclassified",
)

# =========================================================
# Shared legend
# =========================================================
legend = figure.legend(
    handles=class_legend_handles + [error_legend_handle],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=11,

    frameon=True,
    fancybox=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="grey",

    title="Source class",
    fontsize=9,
    title_fontsize=9,

    handletextpad=0.1,
    columnspacing=0.55,
    borderpad=0.2,
    borderaxespad=0.0,
)



# Leave room for the shared axis labels and framed legend.
figure.subplots_adjust(
    top=0.94,
    bottom=0.34,
    left=0.07,
    right=0.99,
    wspace=0.24,
)

# =========================================================
# Save outputs
# =========================================================
comparison_png = (
    RESULT_DIR
    / "figure2_shared_pca.png"
)

comparison_pdf = (
    RESULT_DIR
    / "figure2_shared_pca.pdf"
)

# Publication version: genuine vector PDF generated directly
# from the Matplotlib artists.
figure.savefig(
    comparison_pdf,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.04,
)

# PNG is retained only as a screen-check preview.
figure.savefig(
    comparison_png,
    format="png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.4,
)

plt.show()
plt.close(figure)

print("Saved vector figure:", comparison_pdf)
print("Saved preview figure:", comparison_png)



In [ ]:
pca_metadata = {
    "analysis_fold": int(PROCRUSTES_FOLD),
    "reference_model": REFERENCE_MODEL_NAME,
    "alignment": (
        "orthogonal rotation/reflection only; "
        "no translation and no scaling"
    ),
    "shared_pca_fit": (
        "PCA fitted once to the concatenation "
        "of all aligned model embeddings"
    ),
    "explained_variance_ratio": [
        float(value)
        for value in explained_variance_ratio
    ],
    "cumulative_explained_variance": float(
        explained_variance_ratio.sum()
    ),
    "models": MODEL_SPECS,
}

metadata_path = (
    RESULT_DIR / "figure2_shared_pca_metadata.json"
)
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(
        pca_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved publication outputs:")
for path in [
    RESULT_DIR / "table1_representation_metrics.csv",
    RESULT_DIR / "representation_metrics_by_fold.csv",
    RESULT_DIR / "representation_metrics_std.csv",
    RESULT_DIR / "supplementary_tsne_fold0.pdf",
    RESULT_DIR / "supplementary_tsne_fold0.png",
    RESULT_DIR / "figure2_shared_pca.pdf",
    RESULT_DIR / "figure2_shared_pca.png",
    RESULT_DIR / "procrustes_alignment_diagnostics.csv",
    metadata_path,
]:
    print(" -", path)

elapsed_minutes = (
    time.time() - analysis_start_time
) / 60.0
print(f"\nTotal elapsed time: {elapsed_minutes:.1f} minutes")


## Expected public outputs

The following files are written to `results/diagnostics/`:

```text
table1_representation_metrics.csv
representation_metrics_by_fold.csv
representation_metrics_std.csv
supplementary_tsne_fold0.pdf
supplementary_tsne_fold0.png
figure2_shared_pca.pdf
figure2_shared_pca.png
procrustes_alignment_diagnostics.csv
figure2_shared_pca_metadata.json
```

Large embeddings, probability arrays, rotation matrices, and coordinate caches remain under `ensemble_artifacts/diagnostics/representation_analysis/` and should normally be excluded from Git.
